In [ ]:
# Import libraries
import pandas as pd
from datetime import time
from pathlib import Path

In [ ]:
## Load filtered screenings (output from amc_total_req.ipynb)
amc_screenings_filtered = pd.read_csv(
    '/Users/jackwalker/Data/2026/CUNY_fall/adv-data-assignment-1/data/export/screenings/amc/screenings-filtered-dataset/amc_screenings_relevant_to_law.csv'
)
print(amc_screenings_filtered)

## 3. Check for time of day adherence

Peak hours definition:
- Friday: 5:59 PM–11:01 PM
- Saturday/Sunday: 11:29 AM–11:01 PM

Peak allocation rule:
- At least half of required OC showings in peak hours.
- Exception: if fewer than 1/8 of film showings are in peak hours, then all peak-hour showings must be open captioned.

Weekday evening rule (outside peak):
- For required OC showings outside peak, at least half should be Mon–Thu, starting after 5:59 PM and finishing before 11:01 PM.
- Exception: if fewer than 1/8 of film showings are after 5:59 PM Mon–Thu, then all such showings must be OC.

In [ ]:
# Prepare time-of-day analysis
peak_df = amc_screenings_filtered.copy()

peak_df['start_dt'] = pd.to_datetime(peak_df['Date'] + ' ' + peak_df['Showtime'], errors='coerce')
peak_df['end_dt'] = peak_df['start_dt'] + pd.to_timedelta(pd.to_numeric(peak_df['Runtime minutes'], errors='coerce').fillna(0), unit='m')

fri = peak_df['Day of week'].str.upper().eq('FRIDAY')
wknd = peak_df['Day of week'].str.upper().isin(['SATURDAY', 'SUNDAY'])

fri_overlap = fri & (peak_df['start_dt'] < pd.to_datetime(peak_df['Date'] + ' 11:01 PM')) & (peak_df['end_dt'] > pd.to_datetime(peak_df['Date'] + ' 5:59 PM'))
wknd_overlap = wknd & (peak_df['start_dt'] < pd.to_datetime(peak_df['Date'] + ' 11:01 PM')) & (peak_df['end_dt'] > pd.to_datetime(peak_df['Date'] + ' 11:29 AM'))

peak_df['is_peak_oc'] = peak_df['Open captioned?'].eq('Yes') & (fri_overlap | wknd_overlap)

In [ ]:
# Store the count and rate out of all of the screenings in columns.
peak_oc_summary = peak_df.groupby(['Theater', 'Film']).agg(
    Total_all_screenings=('Film', 'size'),
    Total_peak_screenings=( 'is_peak_oc', lambda s: (fri_overlap | wknd_overlap).loc[s.index].sum() ),
    Peak_OC_count=('is_peak_oc', 'sum')
).assign(
    Peak_OC_rate=lambda d: d['Peak_OC_count'] / d['Total_all_screenings']
).reset_index()

print(peak_oc_summary)

In [ ]:
# Calculate peak screenings rate and boolean for 1/8 threshold
peak_oc_summary['Peak_screenings_rate'] = (
    peak_oc_summary['Total_peak_screenings'] / peak_oc_summary['Total_all_screenings']
)

col = peak_oc_summary.pop('Peak_screenings_rate')
peak_oc_summary.insert(
    peak_oc_summary.columns.get_loc('Total_peak_screenings') + 1,
    'Peak_screenings_rate',
    col
)

# Boolean: fewer than 1/8 of all screenings are in peak hours.
peak_oc_summary['Peak_screenings_rate_below_1_8'] = (
    peak_oc_summary['Peak_screenings_rate'] < (1/8)
)

# Boolean: all peak screenings are open captioned.
peak_oc_summary['All_peak_screenings_OC'] = (
    peak_oc_summary['Peak_OC_count'] == peak_oc_summary['Total_peak_screenings']
)

peak_oc_summary['Peak_OC_violation'] = (
    peak_oc_summary['Total_peak_screenings'].ne(0) &
    peak_oc_summary['Peak_screenings_rate_below_1_8'] &
    ~peak_oc_summary['All_peak_screenings_OC']
)

print(peak_oc_summary.columns.tolist())
print(peak_oc_summary)

In [ ]:
## Check Mon–Thu evening and calculate related booleans
is_peak_any = (fri_overlap | wknd_overlap)
is_non_peak = ~is_peak_any

is_mon_thu = peak_df['Day of week'].str.upper().isin(['MONDAY', 'TUESDAY', 'WEDNESDAY', 'THURSDAY'])
starts_after_559 = peak_df['start_dt'].dt.time > pd.to_datetime('5:59 PM').time()
ends_before_1101 = peak_df['end_dt'].dt.time < pd.to_datetime('11:01 PM').time()

is_mon_thu_evening = is_mon_thu & starts_after_559 & ends_before_1101
is_non_peak_oc = peak_df['Open captioned?'].eq('Yes') & is_non_peak
is_mon_thu_evening_oc = peak_df['Open captioned?'].eq('Yes') & is_mon_thu_evening

non_peak_summary = peak_df.groupby(['Theater', 'Film']).agg(
    Outside_peak_OC_count=('Film', lambda s: is_non_peak_oc.loc[s.index].sum()),
    MonThu_evening_OC_count=('Film', lambda s: is_mon_thu_evening_oc.loc[s.index].sum()),
    Outside_peak_total_screenings=('Film', lambda s: is_non_peak.loc[s.index].sum()),
    MonThu_evening_total_screenings=('Film', lambda s: is_mon_thu_evening.loc[s.index].sum()),
).reset_index()

peak_oc_summary = peak_oc_summary.merge(non_peak_summary, on=['Theater', 'Film'], how='left')

## If that number is more than half of the outside peak showings, store a true value in a new column. If not, store a false value.
peak_oc_summary['MonThu_evening_OC_at_least_half_of_outside_peak_OC'] = (
    peak_oc_summary['MonThu_evening_OC_count'] >= (peak_oc_summary['Outside_peak_OC_count'] / 2)
)

## Calculate whether fewer than 1/8 of a film's overall screenings are after 5:59 PM Mon–Thu, and store that info as a boolean.
peak_oc_summary['MonThu_evening_screenings_rate'] = (
    peak_oc_summary['MonThu_evening_total_screenings'] / peak_oc_summary['Total_all_screenings']
)
peak_oc_summary['MonThu_evening_screenings_rate_below_1_8'] = (
    peak_oc_summary['MonThu_evening_screenings_rate'] < (1/8)
)

## Check whether all qualifying Mon–Thu evening screenings are open captioned.
peak_oc_summary['All_MonThu_evening_screenings_OC'] = (
    peak_oc_summary['MonThu_evening_OC_count'] == peak_oc_summary['MonThu_evening_total_screenings']
)

print(peak_oc_summary)

In [ ]:
# Export time-of-day analysis
base_export_dir = Path('/Users/jackwalker/Data/2026/CUNY_fall/adv-data-assignment-1/data/export/screenings/amc')
violations_by_time_dir = base_export_dir / 'violations-by-time'

peak_oc_summary.to_csv(violations_by_time_dir / 'time_violations.csv', index=False)

print('Exported:')
print('-', violations_by_time_dir / 'time_violations.csv')